# Notebook 01: UCI OPPORTUNITY Dataset Exploration & Verification

**Objective**: Inspect downloaded raw files, column mappings, sensory modalities, and activity labels without making assumptions or modifying raw files.

> **Research Rule**: Never fabricate column headers, sensor mappings, or sampling frequency. Derive all metadata directly from official files.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to path
current = Path.cwd()
REPO_ROOT = current.parent if current.name == "notebooks" else current
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.api as api
from src.data.inspect_opportunity import run_inspection

api.setup_project()

## 1. Run Automated Dataset Audit
Audits all 24 raw `.dat` files, parses official `column_names.txt`, and generates machine-readable metadata in `data/processed/`.

In [ ]:
report = run_inspection()
print(f"Total Files Inspected : {report['total_files']}")
print(f"Total Timesteps       : {report['total_rows']:,}")
print(f"Columns per File      : {report['columns_per_file']}")
print(f"Sampling Frequency    : {report['sampling_rate_hz']} Hz")
print(f"On-Body Channels      : {len(report['sensor_categories']['on_body_sensors'])} channels")

## 2. Inspect a Real Continuous Recording (Subject 1, ADL 1)

In [ ]:
data, labels, timestamps = api.load_opportunity_data(subject="S1", run="ADL1", target_track="Locomotion")

print(f"Recording Shape: {data.shape} (timesteps, on-body channels)")
print(f"Labels Count   : {len(labels)}")
print(f"Sampling Rate  : {1000.0 / np.median(np.diff(timestamps)):.1f} Hz")
print(f"Label Counts   : {pd.Series(labels).value_counts().to_dict()}")

## 3. Visualize Multi-Sensor Inertial Waveforms

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
time_range = slice(1000, 1600)  # ~20 seconds of activity

axes[0].plot(data[time_range, 0:3], label=["Back_Acc_X", "Back_Acc_Y", "Back_Acc_Z"])
axes[0].set_title("Trunk Acceleration")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].plot(data[time_range, 25:28], label=["R_Arm_Acc_X", "R_Arm_Acc_Y", "R_Arm_Acc_Z"])
axes[1].set_title("Right Arm Acceleration")
axes[1].legend(loc="upper right", fontsize=8)

axes[2].plot(labels[time_range], color="red", label="Locomotion Label")
axes[2].set_title("Locomotion Ground Truth (Stand, Walk, Sit, Lie)")
axes[2].set_xlabel("Timestep (30 Hz)")
axes[2].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()